# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates step-by-step loading and exploration of the FAIR² dataset using the `mlcroissant` library. All dataset elements—record sets, fields, columns—are referenced by their `@id` as required for precise, consistent analytics with the Croissant schema.

### Dataset Source
The dataset is defined by a public Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load both the metadata and full records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a Croissant Resource object

print("Dataset Title:", metadata.name)
print("Description:\n", metadata.description[:400], "...\n")  # Print a snippet
print("Identifier:", getattr(metadata, 'identifier', None))
print("Cite as:", getattr(metadata, 'citeAs', None))
print("Date Published:", getattr(metadata, 'datePublished', None))

## 2. Data Overview
Review available record sets, their fields, and the full list of entity `@id`s. This is important for proper referencing throughout the notebook.

In [ ]:
# List all available record sets and their @id, name, and fields
# mlcroissant uses the dataset.metadata.record_sets property for the top-level record sets

if not hasattr(metadata, 'record_sets') or not metadata.record_sets:
    print("No record sets declared in metadata.record_sets. Attempting to discover by introspection.")
    import json
    meta_json = dataset.metadata.to_json()
    # Try to look for any record set by @type==RecordSet
    def find_record_sets(obj):
        if isinstance(obj, dict):
            if obj.get('@type', '').endswith('RecordSet'):
                yield obj
            for v in obj.values():
                yield from find_record_sets(v)
        elif isinstance(obj, list):
            for item in obj:
                yield from find_record_sets(item)
    recordsets = list(find_record_sets(meta_json))
    if not recordsets:
        print("No RecordSet found in metadata. Aborting section.")
    else:
        print(f"Found {len(recordsets)} record set(s):\n")
        for rs in recordsets:
            print(f"RecordSet name: {rs.get('name', None)}")
            print(f"  @id: {rs.get('@id')}")
            print(f"  Fields/columns:")
            if 'field' in rs:
                fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
                for f in fields:
                    print(f"    - {f.get('name', None)} (@id: {f.get('@id')})")
            elif 'column' in rs:
                columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]    
                for c in columns:
                    print(f"    - {c.get('name', None)} (@id: {c.get('@id')})")
            print()

    # Save list of record set IDs to use for extraction
    record_set_ids = [rs['@id'] for rs in recordsets]
else:
    record_set_ids = [rs['@id'] for rs in metadata.record_sets]

print("Discovered Record Set @ids:")
for rsid in record_set_ids:
    print(" -", rsid)

## 3. Data Extraction
Now, we extract the tabular records for each record set and load them into pandas DataFrames. All access is by `@id` for full semantic reference.

In [ ]:
dataframes = {}
# Most datasets have one main record set. We'll pull them all by ID (usually there is only one for tabular data).

for record_set_id in record_set_ids:
    try:
        # Each record is a dict with keys as field @id
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set {record_set_id}. Columns (by @id):\n  " + ", ".join(df.columns))
        # Preview 5 rows
        display(df.head())
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")     

## 4. Exploratory Data Analysis (EDA)

We'll analyze the dataset, performing numeric normalization, filter operations, and simple grouping with all references by `@id`.

**Step 1:** Identify a numeric field to use

**Step 2:** Filter, normalize, group.

In [ ]:
# For demonstration, select the first available record set/DataFrame
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    # Show columns (all are @id references for fields/columns)
    print(f"Columns in {main_record_set_id}:")
    for col in df.columns:
        print(f" - {col}")

    # Try to find a numeric-looking column (e.g. age, or anything with type int or float)
    numeric_field_candidates = []

    # Inspect dtypes for numeric columns
    for col in df.columns:
        try:
            # Try to convert the column to numeric, check if there are finite values
            if pd.to_numeric(df[col], errors='coerce').notnull().sum() > 0:
                numeric_field_candidates.append(col)
        except Exception:
            continue
    
    print("\nPossible numeric fields (by @id):", numeric_field_candidates)

    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()  # Arbitrary threshold: mean value
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field (categorical)
        # Pick a non-numeric field if possible (by @id)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < len(df) / 2 and not pd.api.types.is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable non-numeric group field found.")
    else:
        print("No numeric fields found in first record set.")
else:
    print("No loaded record sets to analyze.")

## 5. Visualization
Visualize one or more distributions or relationships in the data. All plots reference fields by their exact `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If EDA section found numeric_field_id, plot its distribution
if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field_id exists, show boxplot
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(9, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field selected for visualization.")

## 6. Conclusion

This notebook demonstrates how to programmatically discover, load, and explore tabular data from a FAIR² dataset using the `mlcroissant` library, referencing all fields and entities by their schema `@id`. The workflow shown here is reproducible for other Croissant-based datasets, supporting full semantic provenance and field mapping.

**Key observations:**
- The dataset includes comprehensive clinical and molecular variables for cancer survivors with secondary colorectal cancer.
- All analytics directly reference field/column `@id`, ensuring reproducibility and interoperability.
- Basic filtering, normalization, grouping, and visualization are supported using standard pandas and seaborn tools.

For additional analysis or model building, continue working with the DataFrames as shown, always referencing data fields by their `@id`.